In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import warnings; warnings.simplefilter('ignore')
import sys
import h5py
import pandas as pd
import seaborn as sns
sys.path.insert(0, '/Users/jsmonzon/Research/SatGen/mcmc/src/')
import jsm_ancillary
import jsm_visualize
import jsm_SHMR
import jsm_mcmc
import jsm_stats
import jsm_models
import evolve as ev
import galhalo as gh
import profiles as profiles
import config as cfg
import os
import cosmo as co

In [3]:
plt.style.use('../../../SatGen/notebooks/paper1/paper.mplstyle')
double_textwidth = 7.0 #inches
single_textwidth = 3.5 #inches
levelz = [1-0.99, 1-0.95, 1-0.68]

In [22]:
fid = jsm_ancillary.load_massspec("../../data/fid/", "artificial", 0)

satgen_high_mass = fid[fid["logMvir"] > 12.5]
satgen_high_mass = satgen_high_mass.sort_values("logMvir")

In [ ]:
fig, ax = plt.subplots()

fid.logMvir_bincenters = np.unique(fid["logMvir"])

norm = colors.Normalize(
    vmin=fid.logMvir_bincenters.min(),
    vmax=fid.logMvir_bincenters.max()
)
cmap = cm.viridis

for mass_bin in fid.logMvir_bincenters:

    subsample = fid["logMvir"] == mass_bin
    a50s = fid["loga50"][subsample].values
    fsub = fid["logfsub"][subsample].values

    x = -a50s   # = log10(1+zf)

    color = cmap(norm(mass_bin))

    sns.kdeplot(
        x=x,
        y=fsub,
        levels=levelz,
        color=color,
        ax=ax
    )

ax.set_xlim(0, 0.6)
ax.set_ylim(-3.5, 0)
# ---- add the analytic relations ----
xline = np.linspace(*ax.get_xlim(), 200)

ax.plot(xline, -0.48 + (-2.10)*xline, ls="--", lw=2, c="black",
        label=r"1st order")

ax.plot(xline, -1.2 + (-4)*xline, ls=":", lw=2, c="black",
        label=r"2nd order")

# colorbar
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
fig.colorbar(sm, ax=ax, label=r"$\log(M_{\rm vir}/M_\odot)$")

ax.set_xlabel(r"$\log(1+z_f)$")
ax.set_ylabel(r"$\log f_{\rm sub}$")

ax.legend(loc=3)
plt.show


In [ ]:
fid = jsm_ancillary.load_massspec("../../data/fid/", "artificial", 3)
fid = fid[fid["logMvir"] > 12.5]

In [ ]:
fig, ax = plt.subplots()

fid.logMvir_bincenters = np.unique(fid["logMvir"])

norm = colors.Normalize(
    vmin=fid.logMvir_bincenters.min(),
    vmax=fid.logMvir_bincenters.max()
)
cmap = cm.viridis

for mass_bin in fid.logMvir_bincenters:

    subsample = fid["logMvir"] == mass_bin
    a50s = fid["loga50"][subsample].values
    fsub = fid["logfsub"][subsample].values

    x = -a50s   # = log10(1+zf)

    color = cmap(norm(mass_bin))

    sns.kdeplot(
        x=x,
        y=fsub,
        levels=levelz,
        color=color,
        ax=ax
    )

ax.set_xlim(0, 0.6)
ax.set_ylim(-3.5, 0)
# ---- add the analytic relations ----
xline = np.linspace(*ax.get_xlim(), 200)

ax.plot(xline, -0.48 + (-2.10)*xline, ls="--", lw=2, c="black",
        label=r"1st order")

ax.plot(xline, -1.2 + (-4)*xline, ls=":", lw=2, c="black",
        label=r"2nd order")

# colorbar
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
fig.colorbar(sm, ax=ax, label=r"$\log(M_{\rm vir}/M_\odot)$")

ax.set_xlabel(r"$\log(1+z_f)$")
ax.set_ylabel(r"$\log f_{\rm sub}$")

ax.legend(loc=3)
plt.show


In [ ]:
fid = jsm_ancillary.load_massspec("../../data/fid/", "artificial", 1)
fid = fid[fid["logMvir"] > 12.5]

In [ ]:
def dmdt(m, M, tau_dyn):
    """
    Compute dm/dt for subhalo mass loss.

    Parameters
    ----------
    m : float or array
        Subhalo mass.
    M : float or array
        Host halo mass.
    tau_dyn : float or array
        Dynamical time at time t.

    Returns
    -------
    float or array
        Time derivative dm/dt.
    """
    return -0.86 * (m / tau_dyn) * (m / M)**0.07

In [ ]:
early = jsm_visualize.Tree_Vis(file="../../../StellarHalo/data/four_examples/DF/early_evo.npz", merger_crit=-2, fesc=0.2, scatter=False, verbose=False)

In [ ]:
plt.plot(early.CosmicTime, early.ave_mass[44])
plt.plot(early.CosmicTime, early.mass[44])
plt.yscale("log")
plt.show()

In [ ]:
def toy_model(sub_ind):
    m_evo = np.full(cfg.zsample.shape[0], np.nan)

    zsample = cfg.zsample
    tsample = cfg.tsample
    host_mass = early.mass[0]

    # z_acc = early.acc_index[sub_ind]
    z_acc = early.proper_acc_index[sub_ind]

    for z_ind in range(len(zsample)-1, -1, -1):

        z = zsample[z_ind]

        if z_ind > z_acc:
            continue

        elif z_ind == z_acc:
            m_evo[z_ind] = early.acc_mass[sub_ind]

        else:

            m_prev = m_evo[z_ind+1]

            if np.isnan(m_prev):
                continue

            dt = tsample[z_ind] - tsample[z_ind+1]

            #m_lost = dmdt(m_prev, host_mass[z_ind], early.host_profiles[z_ind].tdyn(early.VirialRadius[0, z_ind]))
            m_lost = dmdt(m_prev, host_mass[z_ind], co.tdyn(z))


            m_evo[z_ind] = m_prev + m_lost * dt

    return m_evo[0]/early.acc_mass[sub_ind]

In [ ]:
wow = []
for subhalo in range(early.Nhalo):
    wow.append(toy_model(subhalo))

fb_toy = np.array(wow)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors

fig, ax = plt.subplots()

log1pz = np.log10(1 + early.acc_redshift)

norm = colors.Normalize(
    vmin=log1pz.min(),
    vmax=log1pz.max()
)

sc = ax.scatter(
    early.fb[:, 0],
    fb_toy,
    c=log1pz,
    cmap="viridis",
    norm=norm
)

ax.plot(np.linspace(0,1), np.linspace(0,1), c="black")

ax.set_xscale("log")
ax.set_yscale("log")

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r"$\log(1+z_{\rm acc})$")

ax.set_xlim(1e-4, 1)
ax.set_ylim(1e-4, 1)

ax.set_xlabel("log f$_{\\rm b}$ \n(orbit evo)")
ax.set_ylabel("log f$_{\\rm b}$ \n(average model)")
plt.show()